# GridPulse BR — Bronze Ingestion

## Objective

Build the Bronze layer for the ANEEL Collective Continuity Indicators dataset.

The Bronze layer preserves the source data while adding technical metadata
required for lineage, auditing and reproducibility.

## Source

**Provider:** ANEEL — Brazilian Electricity Regulatory Agency  
**Dataset:** Collective Continuity Indicators  
**Format:** Apache Parquet  
**Landing zone:** Unity Catalog Volume

## Bronze responsibilities

- Preserve all source records and source attributes.
- Preserve the original source values without business transformations.
- Add ingestion metadata for traceability.
- Persist the dataset as a Delta table.
- Validate source-to-target row reconciliation.
- Provide a reliable input for the Silver layer.

## Out of scope

The Bronze layer does **not**:

- deduplicate business records;
- standardize business attributes;
- filter specific indicators;
- apply regulatory rules;
- create analytics metrics.

These transformations belong to downstream layers.

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

SOURCE_SYSTEM = "ANEEL"

SOURCE_FILE = (
    "/Volumes/workspace/gridpulse/landing/aneel/"
    "continuity_indicators/"
    "indicadores-continuidade-coletivos-2020-2029.parquet"
)

TARGET_TABLE = "workspace.gridpulse.bronze_aneel_continuity_indicators"

INGESTION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

print(f"Source file  : {SOURCE_FILE}")
print(f"Target table : {TARGET_TABLE}")
print(f"Ingestion ID : {INGESTION_ID}")

## 1. Read source data

Read the immutable Parquet file from the landing zone.

No business transformations are applied at this stage. The objective is to
preserve the source dataset before persisting it into the Bronze layer.

In [0]:
df_source = spark.read.parquet(SOURCE_FILE)

source_row_count = df_source.count()

print(f"Source rows    : {source_row_count:,}")
print(f"Source columns : {len(df_source.columns)}")

df_source.printSchema()

## 2. Add technical metadata

Technical metadata is added to every record to provide ingestion lineage and
support future auditing and troubleshooting.

Business attributes remain unchanged.

In [0]:
ingested_at = datetime.now(timezone.utc)

df_bronze = (
    df_source
    .withColumn("_ingested_at", F.lit(ingested_at))
    .withColumn("_source_file", F.lit(SOURCE_FILE))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_ingestion_id", F.lit(INGESTION_ID))
)

print(f"Bronze columns : {len(df_bronze.columns)}")

df_bronze.printSchema()

## 3. Pre-write validation

Validate the Bronze DataFrame before persistence.

The pipeline fails early if the source is empty or if the expected technical
metadata columns were not created.

In [0]:
TECHNICAL_COLUMNS = {
    "_ingested_at",
    "_source_file",
    "_source_system",
    "_ingestion_id"
}

missing_metadata = TECHNICAL_COLUMNS - set(df_bronze.columns)

assert source_row_count > 0, "Source dataset is empty."

assert not missing_metadata, (
    f"Missing technical metadata columns: {missing_metadata}"
)

print("Pre-write validation passed.")
print(f"Rows ready for Bronze : {source_row_count:,}")

## 4. Persist Bronze Delta table

Persist the validated source snapshot as a managed Delta table.

### MVP write strategy

This iteration uses a **full-snapshot replacement strategy**.

The ANEEL source is currently delivered as a consolidated Parquet resource.
Appending the complete file on every execution would duplicate previously
ingested records.

For the MVP, the Bronze table therefore represents the latest successfully
ingested source snapshot.

Future iterations will evolve this strategy to support incremental ingestion
and historical source versions.

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Bronze table created successfully: {TARGET_TABLE}")

## 5. Source-to-target reconciliation

Validate that the persisted Bronze table contains the complete source snapshot.

The pipeline must fail if the number of persisted records differs from the
number of records read from the landing source.

In [0]:
df_bronze_persisted = spark.table(TARGET_TABLE)

target_row_count = df_bronze_persisted.count()

print(f"Source rows : {source_row_count:,}")
print(f"Target rows : {target_row_count:,}")

assert source_row_count == target_row_count, (
    f"Reconciliation failed: source={source_row_count:,}, "
    f"target={target_row_count:,}"
)

print("Source-to-target reconciliation passed.")

## 6. Lineage validation

Validate that the persisted snapshot can be traced back to exactly one
ingestion execution and one source system.

In [0]:
lineage_profile = (
    df_bronze_persisted
    .groupBy(
        "_ingestion_id",
        "_source_system",
        "_source_file"
    )
    .agg(
        F.count("*").alias("row_count"),
        F.min("_ingested_at").alias("min_ingested_at"),
        F.max("_ingested_at").alias("max_ingested_at")
    )
)

display(lineage_profile)

## 7. Bronze data quality checks

Validate structural and domain expectations after persistence.

These checks do not modify source data. They verify that the Bronze snapshot
remains consistent with the known characteristics discovered during source
profiling.

In [0]:
quality_metrics = (
    df_bronze_persisted
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("SigAgente").alias("distributors"),
        F.countDistinct("IdeConjUndConsumidoras").alias("consumer_sets"),
        F.countDistinct("SigIndicador").alias("indicators"),
        F.min("AnoIndice").alias("min_year"),
        F.max("AnoIndice").alias("max_year"),
        F.min("NumPeriodoIndice").alias("min_period"),
        F.max("NumPeriodoIndice").alias("max_period")
    )
)

display(quality_metrics)

## 8. Automated quality assertions

Critical expectations are enforced as executable checks.

If a future source snapshot violates these assumptions, the ingestion process
fails explicitly instead of silently publishing unexpected data.

In [0]:
metrics = quality_metrics.first()

assert metrics["row_count"] == source_row_count, (
    "Unexpected Bronze row count."
)

assert metrics["min_year"] >= 2020, (
    f"Unexpected minimum year: {metrics['min_year']}"
)

assert metrics["max_year"] <= 2029, (
    f"Unexpected maximum year: {metrics['max_year']}"
)

assert metrics["min_period"] >= 1, (
    f"Invalid minimum period: {metrics['min_period']}"
)

assert metrics["max_period"] <= 12, (
    f"Invalid maximum period: {metrics['max_period']}"
)

print("All Bronze quality assertions passed.")

## 9. Ingestion summary

Produce a compact execution summary for operational visibility.

In [0]:
print("=" * 60)
print("GRIDPULSE BR — BRONZE INGESTION SUMMARY")
print("=" * 60)

print(f"Source system      : {SOURCE_SYSTEM}")
print(f"Source rows        : {source_row_count:,}")
print(f"Bronze rows        : {target_row_count:,}")
print(f"Source columns     : {len(df_source.columns)}")
print(f"Bronze columns     : {len(df_bronze_persisted.columns)}")
print(f"Distributors       : {metrics['distributors']:,}")
print(f"Consumer sets      : {metrics['consumer_sets']:,}")
print(f"Indicators         : {metrics['indicators']:,}")
print(f"Data coverage      : {metrics['min_year']} - {metrics['max_year']}")
print(f"Ingestion ID       : {INGESTION_ID}")
print(f"Target table       : {TARGET_TABLE}")

print("=" * 60)
print("STATUS: SUCCESS")
print("=" * 60)